In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from dataclasses import dataclass
from anndata import AnnData
import scanpy as sc

In [3]:
# Add parent directory to path
os.chdir("/workspace/")

# Import STAGM
from src.adata_processing import (
    LoadBatch10xAdata,
    LoadSingle10xAdata,
    LoadBatchAdata,
    LoadSingleAdata,
)
from src.stagm import STAGM

In [4]:
# Dataclass for the configuration of the model
@dataclass
class ModelConfig:
    seed: int
    learning_rate: float
    num_hidden: int
    num_proj_hidden: int
    activation: str
    base_model: str
    num_layers: int
    drop_edge_rate_1: float
    drop_edge_rate_2: float
    drop_feature_rate_1: float
    drop_feature_rate_2: float
    tau: int
    tau_decay: float
    num_epochs: int
    weight_decay: float
    num_clusters: int
    num_gene: int
    num_neigh: int
    k: int
    dropout: float
    order_by_degree: bool
    shuffle_ind: int
    d_state: int
    d_conv: int


# Dataclass for the configuration of the dataset
@dataclass
class DatasetConfig:
    dataset: str
    slide: str
    label: bool
    file_list: list[str]

In [5]:
# Initialize the model configuration with the specified parameters
model_config = ModelConfig(
    seed=39788,
    learning_rate=0.0005,
    num_hidden=64,
    num_proj_hidden=64,
    activation="prelu",
    base_model="GCNConv",
    num_layers=1,
    drop_edge_rate_1=None,
    drop_edge_rate_2=None,
    drop_feature_rate_1=0.1,
    drop_feature_rate_2=0.2,
    tau=35,
    tau_decay=None,
    num_epochs=300,
    weight_decay=1e-05,
    num_clusters=7,
    num_gene=3000,
    num_neigh=5,
    k=80,
    dropout=0.0,
    order_by_degree=False,
    shuffle_ind=0,
    d_state=16,
    d_conv=4,
)


# Initialize the dataset configuration with the specified parameters
args = DatasetConfig(
    dataset="visium", slide="Mouse_Brain_Posterior", label=False, file_list=None
)

In [6]:
# Define the base folder and the path to the specific dataset and slide
data_path = Path("./Dataset")
data_path = data_path / args.dataset / args.slide
data_path

PosixPath('Dataset/visium/Mouse_Brain_Posterior')

In [7]:
# Load the AnnData object using the specified parameters
adata: AnnData = LoadSingle10xAdata(
    path=str(data_path),
    n_neighbors=model_config.num_neigh,
    n_top_genes=model_config.num_gene,
    image_emb=True,
    label=args.label,
).run()

/workspace/src/adata_processing.py:56: FutureWarning: Use `squidpy.read.visium` instead.
  self.adata = sc.read_visium(
/opt/conda/envs/stagm-env/lib/python3.11/site-packages/anndata/_core/anndata.py:1880: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/envs/stagm-env/lib/python3.11/site-packages/anndata/_core/anndata.py:1880: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
Calculating edge_probabilities: 100%|██████████| 3355/3355 [00:00<00:00, 75477.18it/s]

adata load done


In [8]:
# Initialize the STAGM model with the specified parameters and assign the loaded AnnData object to it
stagm = STAGM(args=args, config=model_config, single=False, refine=False)
stagm.adata = adata

In [9]:
stagm.train()

=== prepare for training ===
=== train ===


100%|██████████| [00:07<00:00, 42.68it/s]


In [10]:
# Evaluate the STAGM model
stagm.eva()

=== load ===

========== Model Diagnostics ==========
  Total parameters:     237,825
  Trainable parameters: 237,825
  Model weight size:    0.91 MB
  Peak GPU memory:      729.54 MB
  Training time:        0m 7.0s  (7.0s total)
  Time per epoch:       23.4 ms

  Loss curve:            first=8.7861  last=8.7626  min=8.7626  max=8.7861

  Embedding shape:      (3355, 64)
  Embedding norm — mean=5.6426  std=2.1708  min=2.0559  max=12.2097

[[-0.03003522  0.60498244 -0.15295818 ...  0.25157157  0.4715339
  -0.335497  ]
 [-0.04409402 -0.802275    0.12794174 ... -0.21923015 -0.47614282
   0.12306312]
 [-0.11983686  0.76769304  0.04759558 ...  0.06693367  0.6259572
  -0.39919835]
 ...
 [-0.04800562 -1.2150022   0.26160252 ... -0.3633155  -0.80820096
   0.22138762]
 [-0.06343069 -0.93134034  0.27557945 ... -0.38520426 -0.51647985
   0.0964371 ]
 [ 0.20586807 -0.16121066 -0.08834137 ... -0.07562815 -0.4662621
   0.41705567]]
embedding generated, go clustering


In [11]:
# Perform clustering using the trained STAGM model
stagm.cluster(args.label)

Searching resolution...
Cluster count 12 is too large, adjusting end downward...
resolution=0.1650, cluster number=9
resolution=0.1600, cluster number=9
resolution=0.1550, cluster number=9
resolution=0.1500, cluster number=9
resolution=0.1450, cluster number=9
resolution=0.1400, cluster number=9
resolution=0.1350, cluster number=9
resolution=0.1300, cluster number=8
resolution=0.1250, cluster number=8
resolution=0.1200, cluster number=8
resolution=0.1150, cluster number=8
resolution=0.1100, cluster number=8
resolution=0.1050, cluster number=8
resolution=0.1000, cluster number=8
resolution=0.0950, cluster number=6
resolution=0.0900, cluster number=6
resolution=0.0850, cluster number=6
resolution=0.0800, cluster number=6
resolution=0.0750, cluster number=6
resolution=0.0700, cluster number=5
resolution=0.0650, cluster number=5
resolution=0.0600, cluster number=5
resolution=0.0550, cluster number=4
resolution=0.0500, cluster number=5
resolution=0.0450, cluster number=4
resolution=0.0400, 

AssertionError: Resolution not found. Please try a bigger range or a smaller step.

In [ ]:
print(stagm.adata.obs.columns)

Index(['in_tissue', 'array_row', 'array_col', 'leiden'], dtype='object')
